In [1]:
import os
from tools import load_posterior, plot_true_vals, set_boundaries
import numpy as np
import torch
import pandas as pd
import seaborn as sns
import zuko
from zuko.flows import NSF
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import getdist
from getdist import plots as gd_plots
import csv
from copy import deepcopy

from tools import torch_device
device = torch_device()

%matplotlib inline

CUDA is not available. Using CPU.


Load pre-trained networks and test points

In [ ]:
wdir = '/anvil/scratch/x-mho1/cmass-ili'

cosmonames = [r'$\Omega_m$', r'$\Omega_b$', r'$h$', r'$n_s$', r'$\sigma_8$']
hodnames = [r'$\alpha$', r'$\log M_0$', r'$\log M_1$',
            r'$\log M_{\min}$', r'$\sigma_{\log M}$']
noisenames = [r'$\sigma_{\rm radial}$', r'$\sigma_{\rm tangential}$',]
names = cosmonames + noisenames  # +hodnames

# Specify model configuration
nbody = 'quijotelike'
sim = 'fastpm_varnoise'
save_dir = os.path.join(wdir, nbody, sim, 'models')

# Specify data dtype
tracer = 'simbig_lightcone'
# summaries = ['nbar', 'zPk0']  # , 'zPk2', 'zPk4',  'zQk0']
summaries = ['nbar', 'Pk0', 'Pk2', 'Pk4' , 'Qk0']
summary = '+'.join(summaries)
kmin, kmax = 0.0, 0.4
modelpath = os.path.join(save_dir, tracer, summary, f'kmin-{kmin}_kmax-{kmax}')
print(
    f'Loading model: nbody={nbody}, sim={sim}, tracer={tracer}, \n\tsummary={summary}, kmin={kmin}, kmax={kmax}')
print(modelpath)
print('\n'.join(os.listdir(modelpath)))

posterior = load_posterior(modelpath)
for p in posterior.posteriors:
    print(type(p.nde.flow))
    
xtest = np.load(os.path.join(modelpath, 'x_test.npy'))
ytest = np.load(os.path.join(modelpath, 'theta_test.npy'))
    
yrange = np.stack([
    ytest.min(axis=0),
    ytest.max(axis=0)
], axis=1)

name_dict = {
    r'$\Omega_m$':'Omega_m', 
    r'$\Omega_b$':'Omega_b', 
    r'$h$':'h', 
    r'$n_s$':'n_s', 
    r'$\sigma_8$':'sigma_8',
    r'$\alpha$':'alpha', 
    r'$\log M_0$':'logM0', 
    r'$\log M_1$':'logM1', 
    r'$\log M_{\min}$':'logMmin', 
    r'$\sigma_{\log M}$':'sigma_logM',
    r'$\sigma_{\rm radial}$':'sigma_radial', 
    r'$\sigma_{\rm tangential}$':'sigma_tangential',
}
par_names = [name_dict[n] for n in names]

Predict on a random test point

In [ ]:
ind = 10
x0 = torch.Tensor(xtest[ind]).to(device)
y0 = ytest[ind]
samp0 = posterior.sample(x=x0, shape=(5000,))

# Check uniform priors
for n, v in uniform_priors.items():
    if n in par_names:
        i = par_names.index(n)
        prior_mask = (samp0[:,i] >= v[0]) & (samp0[:,i] <= v[1])
        samp0 = samp0[prior_mask]
print('Number of remaining samples:', samp0.shape[0])

Plot the samples

In [ ]:
g = sns.pairplot(
    pd.DataFrame(samp0, columns=names),
    vars=names,
    kind='hist',
    corner=True,
    height=1.5
    # plot_kws={'alpha': 0.5, 'levels': [0.05, 0.36, 1], 'fill': True},  # for kde plot
)

plot_true_vals(g, y0, color='r', lw=1)
# set_boundaries(g, yrange)

Extract the priors for this run

In [ ]:
# Store uniform parameter details
uniform_priors = {
    'Omega_m': [0.1, 0.5],
    'Omega_b': [0.03, 0.07],
    'h': [0.5, 0.9],
    'n_s': [0.8, 1.2],
    'sigma_8': [0.6, 1.0]
}

with open(os.path.join(modelpath,'hodprior.csv'), newline='') as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        if row[1].strip() == 'uniform':
            name = row[0].strip()
            min_val = float(row[2])
            max_val = float(row[3])
            uniform_priors[name] = [min_val, max_val]

print(uniform_priors)



Obtain NF approximation to this posterior

In [ ]:
# Neural spline flow (NSF) with 3 sample features and 5 context features
flow = NSF(samp0.shape[1], transforms=3, hidden_features=[128] * 3)

# Train to maximize the log-likelihood
optimizer = torch.optim.Adam(flow.parameters(), lr=1e-3)

nepoch = 200
train_frac = 0.8

all_train_loss = []
all_val_loss = []

train_set = TensorDataset(samp0[:int(train_frac*samp0.shape[0]),:])
val_set = TensorDataset(samp0[int(train_frac*samp0.shape[0]):,:])

train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1024)

for epoch in tqdm(range(nepoch)):
    flow.train()
    for batch, in train_loader:
        loss = -flow().log_prob(batch).mean()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    all_train_loss.append(loss.detach().numpy())

    # Validation evaluation
    flow.eval()
    with torch.no_grad():
        val_loss = 0.0
        count = 0
        for batch, in val_loader:
            val_loss += -flow().log_prob(batch).sum().item()
            count += batch.size(0)
        val_loss /= count
    all_val_loss.append(val_loss)
        # print(f"[Epoch {epoch}] Val NLL: {val_loss:.4f}")
    
plt.plot(all_train_loss, label='Training')
plt.plot(all_val_loss, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()

In [ ]:
samp1 = flow().sample((5000,))

cosmonames = [r'$\Omega_m$', r'$\Omega_b$', r'$h$', r'$n_s$', r'$\sigma_8$']
hodnames = [r'$\alpha$', r'$\log M_0$', r'$\log M_1$',
            r'$\log M_{\min}$', r'$\sigma_{\log M}$']
noisenames = [r'$\sigma_{\rm radial}$', r'$\sigma_{\rm tangential}$',]

labels = [n[1:-1] for n in names]

# Check uniform priors
for n, v in uniform_priors.items():
    if n in par_names:
        i = par_names.index(n)
        prior_mask = (samp1[:,i] >= v[0]) & (samp1[:,i] <= v[1])
        samp1 = samp1[prior_mask]
print('Number of remaining samples:', samp1.shape[0])


samp0_gd = getdist.MCSamples(samples=samp0.numpy().astype(np.float32), names=par_names, labels=labels, ranges=uniform_priors, label="Samples")
samp1_gd = getdist.MCSamples(samples=samp1.numpy().astype(np.float32), names=par_names, labels=labels, ranges=uniform_priors, label="NF")
g = gd_plots.get_subplot_plotter(width_inch=10)
g.triangle_plot([samp0_gd, samp1_gd], filled=True)
plt.show()

Obtian NN approximation to posterior conditioned to sigma_radial and sigma_tangential

In [ ]:
train_frac = 0.8
nepoch = 500
patience = 50
min_delta = 1e-4
best_val_loss = float('inf')
best_model_state = None
epochs_no_improve = 0
scheduler_patience = 20
scheduler_factor = 0.5
lr = 1e-3
min_lr = 1e-6

# Neural spline flow (NSF)
con_flow = zuko.flows.NSF(samp0.shape[1] - 2, 2, transforms=3, hidden_features=(64, 64))

# Train to maximize the log-likelihood
optimizer = torch.optim.Adam(con_flow.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=scheduler_patience, factor=scheduler_factor,min_lr=min_lr,)

m = np.array([p in ['sigma_radial', 'sigma_tangential'] for p in par_names], dtype=bool)
X_train = samp0[:int(train_frac * samp0.shape[0]), ~m]
C_train = samp0[:int(train_frac * samp0.shape[0]), m]
X_val = samp0[int(train_frac * samp0.shape[0]):, ~m]
C_val = samp0[int(train_frac * samp0.shape[0]):, m]

# Build datasets
train_dataset = TensorDataset(X_train, C_train)
val_dataset = TensorDataset(X_val, C_val)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=1024)

all_train_loss = []
all_val_loss = []

for epoch in tqdm(range(nepoch)):
    con_flow.train()
    for x, c in train_loader:
        loss = -con_flow(c).log_prob(x.unsqueeze(0)).mean()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    all_train_loss.append(loss.detach().numpy())

    # Validation evaluation
    con_flow.eval()
    with torch.no_grad():
        val_loss = 0.0
        count = 0
        for x, c in val_loader:
            val_loss += -con_flow(c).log_prob(x.unsqueeze(0)).sum().item()
            count += x.size(0)
        val_loss /= count
    all_val_loss.append(val_loss)
    
    scheduler.step(val_loss)
    
    # Early stopping check
    if best_val_loss - val_loss > min_delta:
        best_val_loss = val_loss
        best_model_state = deepcopy(con_flow.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

# Load best model
if best_model_state is not None:
    print('Loading state')
    con_flow.load_state_dict(best_model_state)
    
plt.plot(all_train_loss, label='Training')
plt.plot(all_val_loss, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
c = torch.Tensor(y0[m])
samp2 = con_flow(c).sample((5000,))

# Check uniform priors
for n, v in uniform_priors.items():
    if n in par_names:
        i = par_names.index(n)
        prior_mask = (samp2[:,i] >= v[0]) & (samp2[:,i] <= v[1])
        samp2 = samp2[prior_mask]
print('Number of remaining samples:', samp2.shape[0])

labels = [n[1:-1] for n in names]
new_labels = [ell[1:-1] for i, ell in enumerate(names) if not m[i]]
new_names = [ell for i, ell in enumerate(par_names) if not m[i]]

samp0_gd = getdist.MCSamples(samples=samp0.numpy().astype(np.float32), names=par_names, labels=labels, ranges=uniform_priors, label="Samples")
samp2_gd = getdist.MCSamples(samples=samp2.numpy().astype(np.float32), names=new_names, labels=new_labels, ranges=uniform_priors, label="Conditioned")

true_par = {n:v for n,v in zip(par_names, y0)}

g = gd_plots.get_subplot_plotter(width_inch=10)
g.triangle_plot([samp0_gd, samp2_gd], 
                filled=True, 
                markers=true_par, 
                marker_args={'lw': 1, 'c':'k'},
                params=new_names
)
g.export(f'figs/quijotelike_condition_noise_{ind}.png')
plt.show()

# Prepare data for testing

In [ ]:
print(modelpath)
ids_test = np.load(os.path.join(modelpath, 'ids_test.npy'))
theta_test = np.load(os.path.join(modelpath, 'theta_test.npy'))

print(len(ids_test), len(set(ids_test)))

sigma_radial = theta_test[:,5]
plt.plot(theta_test[:,5], theta_test[:,6], '.')
plt.plot(theta_test[::5,5], theta_test[::5,6], '.')
plt.xlabel(r'$\sigma_{\rm radial} \ / \ h^{-1} \, {\rm Mpc}$')
plt.ylabel(r'$\sigma_{\rm transverse} \ / \ h^{-1} \, {\rm Mpc}$')
plt.show()

Check loss curves

In [ ]:
i = 0
dirname = '/anvil/scratch/x-dbartlett/cmass/quijotelike/condition_on_sigma'
summaries = os.listdir(dirname)
summaries.sort()

fig, axs = plt.subplots(2, len(summaries)//2, figsize=(6*len(summaries)//2, 7), sharex=True, sharey=True)
axs[0,0].set_ylabel('Loss')
axs[1,0].set_ylabel('Loss')
axs = np.atleast_2d(axs).flatten()
for ax, summ in zip(axs, summaries):
    fname = os.path.join(dirname, summ, 'kmin-0.0_kmax-0.4', f'train_loss_{i}.npz')
    if not os.path.isfile(fname):
        continue
    data = np.load(fname)
    ax.plot(data['train'], label='Train')
    ax.plot(data['val'], label='Validation')
    ax.legend()
    ax.set_xlabel('Epoch')
    ax.set_title(summ)
fig.tight_layout();
plt.show();

# Plot distribution of biases

In [ ]:
dirname = '/anvil/scratch/x-dbartlett/cmass/quijotelike/condition_on_sigma'
summary = 'nbar+Pk0'

summ_dir = os.path.join(dirname, summ, 'kmin-0.0_kmax-0.4')
files = [os.path.join(summ_dir, f) for f in os.listdir(summ_dir) if f.startswith('samples')]

cosmonames = [r'$\Omega_m$', r'$\Omega_b$', r'$h$', r'$n_s$', r'$\sigma_8$']
hodnames = [r'$\alpha$', r'$\log M_0$', r'$\log M_1$',
            r'$\log M_{\min}$', r'$\sigma_{\log M}$']
noisenames = [r'$\sigma_{\rm radial}$', r'$\sigma_{\rm tangential}$',]
names = cosmonames + noisenames  # +hodnames
name_dict = {
    r'$\Omega_m$':'Omega_m', 
    r'$\Omega_b$':'Omega_b', 
    r'$h$':'h', 
    r'$n_s$':'n_s', 
    r'$\sigma_8$':'sigma_8',
    r'$\alpha$':'alpha', 
    r'$\log M_0$':'logM0', 
    r'$\log M_1$':'logM1', 
    r'$\log M_{\min}$':'logMmin', 
    r'$\sigma_{\log M}$':'sigma_logM',
    r'$\sigma_{\rm radial}$':'sigma_radial', 
    r'$\sigma_{\rm tangential}$':'sigma_tangential',
}
par_names = [name_dict[n] for n in names]

print(par_names)
percs = np.empty((len(files), 3, len(par_names)-2))
truths = np.empty((len(files), len(par_names)-2))
sigma = np.empty((len(files), 2))

for i, f in enumerate(files):
    data = np.load(f)
    y0 = data['y0']
    percs[i] = np.percentile(data['samples'], [50, 16, 84], axis=0)
    truths[i] = y0[:-2]
    sigma[i] = y0[-2:]
    
pull = (percs[:,0,:] - truths) / (percs[:,2,:] - percs[:,0,:])


fig, axs = plt.subplots(truths.shape[1], 2, figsize=(15,15))
for i in range(truths.shape[1]):
    axs[i,0].errorbar(truths[:,i], percs[:,0,i], yerr=[percs[:,0,i]-percs[:,1,i], percs[:,2,i]-percs[:,0,i]], c='k', fmt='.', elinewidth=1)
    axs[i,1].hist(pull[:,i], bins=30, density=True)
    x = np.linspace(-5, 5, 100)
    y = np.exp(-x**2/2) / np.sqrt(2 * np.pi)
    axs[i,1].plot(x, y, color='k')
    axs[i,0].set_title(names[i]);
    axs[i,0].set_title(names[i]);
fig.tight_layout();
plt.show()

# To Do

* Generate mock data
    * Obtain the IDs for all the test sims
    * Pick a single HOD realisation for each one
    * With this realisation, pick a few sigma, evenly spaced in 2D plane [0, 2, 4, 6, 8]: 5^2 = 25 realisations
    * Get summaries for each of these
* Actually, why not just use the pre-run data as we have 1000 for each?
* Summarise result into a bias on the parameters
    * Check the definition of pull above